# Macrocosm photo-z CNN
Thin runner: this notebook **only downloads data and calls the entry point** in `photoz_cnn.py`.
All model / training logic lives in `photoz_cnn.py` (edit it to change the model). 3-fold OOF
outlier finding lives in `cv_outliers.py`.

## 1. Setup + data
Clone the repo (brings `photoz_cnn.py`, `eval.py`, `cv_outliers.py`, `splits/`),
install deps, log into Google, download the catalog.

In [ ]:
![ -d /content/CNN-Model ] || git clone https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /content/CNN-Model
%cd /content/CNN-Model
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/
DATA_DIR = '/content/data'

Download the image shards (~23 GB for all 100; 64px full needs a **High-RAM** runtime).

In [ ]:
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/images_*.npy /content/data/

## 2. Train
Edit `photoz_cnn.py` to change the model. Paste your MLflow token (ask the team) to log the
run + the **50k-val outlier objids**; without a token it just trains. `train()` loads data into
RAM, trains, scores the fixed 50k val, and (with a token) logs params/metrics/outliers to MLflow.

In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir='/content/data',
    crop=64,            # 64 -> High-RAM runtime; 32 -> full 600k fits standard
    N=None,             # cap #train images (e.g. 150000) if RAM-limited; None = all
    batch=256, lr=3e-4, epochs=50,
    run_name='my-run',
    mlflow_token='PASTE_MLFLOW_API_TOKEN_HERE',
)
print(metrics)

## 3. (optional) 3-fold OOF outliers → GCS
Trains 3 models (heavy; loads all 550k @64px ~22 GB → High-RAM). `seed` decides the fold split.
Results go to `gs://macrocosm-lewagon/results/cv_outliers/seed_<seed>/`.

In [ ]:
# from cv_outliers import run
# df = run(seed=0, data_dir='/content/data', crop=64)
# -- or as a script --
# !python cv_outliers.py --seed 0 --data-dir /content/data --out gs://macrocosm-lewagon/results/cv_outliers